In [ ]:
# Prepare paired dataset in Google Drive for this repo
# - Expects source structure like MyDrive/LUTor/Adobe5kJPG/{raw,a,b,c,d,e}
# - Produces MyDrive/LUTor/datasets/Adobe5kJPG_<style>/{train,val}/{input,gt}
# - Uses symlinks when possible (fast, no extra storage); falls back to copy on error

import os
import shutil
from pathlib import Path
from typing import Tuple

try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive', force_remount=False)

# User-configurable
STYLE = 'c'  # choose from 'a','b','c','d','e'
VAL_RATIO = 0.05  # 5% for validation, change as needed
RANDOM_SEED = 42

# Paths
DRIVE_ROOT = Path('/content/drive/MyDrive') if IN_COLAB else Path.home() / 'MyDrive'
PROJECT_ROOT = DRIVE_ROOT / 'LUTor'
SRC_ROOT = PROJECT_ROOT / 'Adobe5kJPG'
DATASETS_ROOT = PROJECT_ROOT / 'datasets'
DST_ROOT = DATASETS_ROOT / f'Adobe5kJPG_{STYLE}'

SRC_RAW = SRC_ROOT / 'raw'
SRC_STYLE = SRC_ROOT / STYLE

for p in [PROJECT_ROOT, SRC_ROOT, SRC_RAW, SRC_STYLE]:
    if not p.exists():
        raise FileNotFoundError(f'Missing path: {p}. Please ensure your Drive has the expected structure.')

# Collect image files (common extensions)
EXTS = {'.jpg','.jpeg','.png','.tif','.tiff'}
raw_files = sorted([p for p in SRC_RAW.iterdir() if p.suffix.lower() in EXTS])
style_files = sorted([p for p in SRC_STYLE.iterdir() if p.suffix.lower() in EXTS])

# Build name -> path map and intersect
raw_map = {f.name: f for f in raw_files}
style_map = {f.name: f for f in style_files}
common_names = sorted(set(raw_map.keys()) & set(style_map.keys()))
if not common_names:
    raise RuntimeError('No paired files found between raw/ and style folder. Filenames must match.')

# Split into train/val deterministically
import random
random.seed(RANDOM_SEED)
names = common_names.copy()
random.shuffle(names)
val_count = max(1, int(len(names) * VAL_RATIO))
val_names = set(names[:val_count])
train_names = set(names[val_count:])

# Create destination structure
for sub in ['train/input','train/gt','val/input','val/gt']:
    (DST_ROOT / sub).mkdir(parents=True, exist_ok=True)


def _link_or_copy(src: Path, dst: Path) -> None:
    if dst.exists():
        return
    try:
        # Prefer relative symlink for portability
        rel_src = os.path.relpath(src, start=dst.parent)
        dst.symlink_to(rel_src)
    except Exception:
        shutil.copy2(src, dst)

# Populate
for split, name_set in [('train', train_names), ('val', val_names)]:
    for name in sorted(name_set):
        src_in = raw_map[name]
        src_gt = style_map[name]
        dst_in = DST_ROOT / split / 'input' / name
        dst_gt = DST_ROOT / split / 'gt' / name
        _link_or_copy(src_in, dst_in)
        _link_or_copy(src_gt, dst_gt)

print(f'Dataset prepared at: {DST_ROOT}')
print('Structure: train/val with input/gt. You can now point training --data_root to this path.')



# LoR-IA-3DLUT • Colab Quickstart

按顺序执行每个单元即可开始训练。

In [ ]:
!nvidia-smi
import os

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/lor_ia3dlut_starter

In [ ]:
!pip install -r requirements.txt

## 准备数据
将你的数据整理为 `/dataset_root/{train,val}/{input,gt}` 结构，然后设置到下面变量。

In [ ]:
DATA_ROOT = '/content/drive/MyDrive/datasets/FiveK_paired'  # 修改成你的路径
WORK_DIR = 'runs/fivek_lor'

In [ ]:
%%bash
python train.py \
  --data.root "$DATA_ROOT" \
  --work_dir "$WORK_DIR" \
  --cfg config/default.yaml

## 验证与可视化

In [ ]:
%%bash
python evaluate.py \
  --data.root "$DATA_ROOT/val" \
  --ckpt "$WORK_DIR/best.ckpt" \
  --out_dir "$WORK_DIR/val_vis"